In [1]:
import torch
print("GPU tersedia:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Nama GPU:", torch.cuda.get_device_name(0))

GPU tersedia: True
Nama GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [2]:
import pandas as pd
import numpy as np

from datasets import Dataset
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB

from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer
)

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import joblib
import os


c:\Users\andyc\Documents\a_skripsi\training\prethesis\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def train_intent_model():
    print("Membaca dataset chat_dataset2.csv...")
    base_dir = os.getcwd()
    dataset_path = os.path.join(base_dir, "data", "chat_dataset2.csv")
    
    if not os.path.exists(dataset_path):
        print("Dataset tidak ditemukan! Tunggu generate.py selesai dulu ya.")
        return None
        
    df = pd.read_csv(dataset_path)
    df = df.dropna()
    
    X = df['teks_chat']
    y = df['label_intent']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    print(f"Total data latih: {len(X_train)} | Total data uji: {len(X_test)}")
    
    print("Melatih model NLU (TF-IDF + SVM)...")
    model = make_pipeline(TfidfVectorizer(), SVC(kernel='linear', probability=True))
    
    model.fit(X_train, y_train)
    
    print("\n--- Hasil Ujian Model (Evaluasi) ---")
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))
    
    os.makedirs("models", exist_ok=True)
    joblib.dump(model, "models/intent_classifier.pkl")
    print("Model berhasil disimpan di models/intent_classifier.pkl\n")
    
    return model

train_intent_model()

In [ ]:
def train_intent_model_nb():
    print("Membaca dataset chat_dataset2.csv...")
    
    base_dir = os.getcwd()
    dataset_path = os.path.join(base_dir, "data", "chat_dataset2.csv")    
    model_dir = os.path.join(base_dir, "models")
    
    if not os.path.exists(dataset_path):
        print(f"Dataset tidak ditemukan di: {dataset_path}")
        return None
        
    df = pd.read_csv(dataset_path)
    df = df.dropna()
    
    X = df['teks_chat']
    y = df['label_intent']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Total data latih: {len(X_train)} | Total data uji: {len(X_test)}")
    
    print("Melatih model NLU (TF-IDF + Naive Bayes)...")
    model = make_pipeline(TfidfVectorizer(), MultinomialNB())
    model.fit(X_train, y_train)
    
    print("\n--- Hasil Uji Model Naive Bayes ---")
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))
    
    os.makedirs("models", exist_ok=True)
    joblib.dump(model, "models/intent_classifier_nb.pkl")
    print(f"Model berhasil disimpan di models/intent_classifier_nb.pkl\n")
    
    return model

train_intent_model_nb()

In [3]:
def train_intent_model_transformer():
    print("Membaca dataset chat_dataset2.csv...")
    base_dir = os.getcwd()
    dataset_path = os.path.join(base_dir, "data", "chat_dataset2.csv")
        
    if not os.path.exists(dataset_path):
        print(f"Dataset tidak ditemukan di: {dataset_path}")
        return None

    df = pd.read_csv(dataset_path).dropna()
    
    label_encoder = LabelEncoder()
    df['label'] = label_encoder.fit_transform(df['label_intent'])
    num_labels = len(label_encoder.classes_)
    
    df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
    print(f"Total data latih: {len(df_train)} | Total data uji: {len(df_test)}")
    
    train_dataset = Dataset.from_pandas(df_train[['teks_chat', 'label']])
    test_dataset = Dataset.from_pandas(df_test[['teks_chat', 'label']])
    
    model_name = "indobenchmark/indobert-base-p1"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    def tokenize_function(examples):
        return tokenizer(examples["teks_chat"], padding="max_length", truncation=True, max_length=128)
    
    train_dataset = train_dataset.map(tokenize_function, batched=True)
    test_dataset = test_dataset.map(tokenize_function, batched=True)
    
    print("Mengunduh/Memuat model IndoBERT...")
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
    
    training_args = TrainingArguments(
        output_dir="./models/transformer_results",
        eval_strategy="epoch",  
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,     
        weight_decay=0.01,
    )
    
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        return {"accuracy": (predictions == labels).mean()}
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics
    )
    
    print("Mulai melatih model Transformer...")
    trainer.train()
    
    print("\n--- Hasil Uji Model Transformer ---")
    predictions = trainer.predict(test_dataset)
    y_pred = np.argmax(predictions.predictions, axis=-1)
    y_true = test_dataset["label"]
    
    target_names = label_encoder.classes_
    print(classification_report(y_true, y_pred, target_names=target_names))
    
    model_save_path = "models"
    tokenizer.save_pretrained(model_save_path)
    model.save_pretrained(model_save_path)
    
    joblib.dump(label_encoder, f"{model_save_path}/label_encoder.pkl")
    
    print(f"Model berhasil disimpan di folder: {model_save_path}\n")
    return model, tokenizer, label_encoder

train_intent_model_transformer()


Membaca dataset chat_dataset2.csv...
Total data latih: 650 | Total data uji: 163


Map: 100%|██████████| 163/163 [00:00<00:00, 9087.51 examples/s]


Mengunduh/Memuat model IndoBERT...


[transformers] You passed `num_labels=8` which is incompatible to the `id2label` map of length `5`.


ValueError: Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434

In [ ]:
def predict_intent(chat_text):
    model_path = "models/intent_classifier.pkl"
    if not os.path.exists(model_path):
        model = train_intent_model()
    else:
        model = joblib.load(model_path)
    
    # Prediksi intent dari chat baru
    prediksi = model.predict([chat_text])[0]
    
    # Ambil nilai probabilitas/keyakinan model (dalam persentase)
    probabilitas = max(model.predict_proba([chat_text])[0]) * 100
    
    return prediksi, probabilitas

In [ ]:
train_intent_model_transformer()
# train_intent_model_nb()
# train_intent_model()
    
# print("--- SIMULASI TESTING AI 1 DI DALAM GAME ---")
# test_chats = [
#     "Jagain rumah gw pak pol, gw bayar mahal nih pake koin",
#     "Lu curigaan mulu sama gw anjir, gw cuma warga biasa",
#     "Woy si Budi dari tadi diem aja, fix dia ketuanya"
# ]

# for chat in test_chats:
#     intent, prob = predict_intent(chat)
#     print(f"Chat Player: '{chat}'")
#     print(f" > AI 1 Menebak: [{intent.upper()}] (Tingkat Keyakinan: {prob:.2f}%)\n")